# HeritageNet — Interactive Model Dashboard

Interactive, live-computed analysis of the champion model (hover, zoom, pan on every chart).
All numbers computed fresh from the checkpoint + test set.

> **Restart & Run All.** Then hover over any chart for details.

In [ ]:
# 1 · Setup ------------------------------------------------------------
import os
if os.path.basename(os.getcwd())=="notebooks": os.chdir("..")

import numpy as np, torch
import torch.nn.functional as F
from pathlib import Path
from collections import Counter
from torch.utils.data import DataLoader
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, confusion_matrix)
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio

from heritagenet.data.datamodule import HeritageDataModule
from heritagenet.models.factory import build_model
from heritagenet.utils.device import get_device

# ---- clean light palette + template ----
INK="#243b53"; SUB="#627d98"; PAPER="#ffffff"; PANEL="#f7f5f2"; GRID="#e9e5df"
TEAL="#1f8a80"; CORAL="#e8663d"; GOLD="#e2a63b"; PLUM="#7b5ea7"; SKY="#2d7dd2"
QUAL=["#d64545","#e8813a","#e2a63b","#3a9d8f"]  # bad→good ramp
def qcolor(v): return QUAL[0] if v<0.6 else QUAL[1] if v<0.75 else QUAL[2] if v<0.9 else QUAL[3]

TEMPLATE=go.layout.Template()
TEMPLATE.layout=go.Layout(
    paper_bgcolor=PAPER, plot_bgcolor=PANEL, font=dict(family="Inter, Segoe UI, sans-serif",
    color=INK, size=13), title=dict(font=dict(size=19,color=INK)),
    xaxis=dict(gridcolor=GRID, zerolinecolor=GRID, linecolor=GRID),
    yaxis=dict(gridcolor=GRID, zerolinecolor=GRID, linecolor=GRID),
    colorway=[TEAL,CORAL,GOLD,PLUM,SKY], margin=dict(l=70,r=30,t=70,b=60),
    hoverlabel=dict(bgcolor="white", font_size=13, bordercolor=GRID))
pio.templates["heritage"]=TEMPLATE
pio.templates.default="heritage"

CKPT='checkpoints/phase2_86acc_FINAL.pt'; MODEL='mobilenet_v3_small'
device=get_device()
dm=HeritageDataModule(root='data',batch_size=32).setup(); classes=dm.classes
model=build_model(MODEL,num_classes=dm.num_classes,pretrained=False,freeze_backbone=False)
st=torch.load(CKPT,map_location=device); model.load_state_dict(st.get('model_state',st))
model.to(device).eval()
print(f'loaded {CKPT} | {dm.num_classes} classes | {len(dm.test_dataset)} test images')

In [ ]:
# predictions once ------------------------------------------------------
loader=DataLoader(dm.test_dataset,batch_size=32,shuffle=False,num_workers=0)
P=[]; y_true=[]
with torch.no_grad():
    for imgs,lbls in loader:
        P.append(F.softmax(model(imgs.to(device)),1).cpu()); y_true+=lbls.tolist()
probs=torch.cat(P).numpy(); y_true=np.array(y_true); y_pred=probs.argmax(1)
pred_conf=probs.max(1); true_conf=probs[np.arange(len(y_true)),y_true]
prec=precision_score(y_true,y_pred,average=None,zero_division=0)
rec =recall_score(y_true,y_pred,average=None,zero_division=0)
f1c =f1_score(y_true,y_pred,average=None,zero_division=0)
counts=Counter(dm.train_dataset.targets); sizes=np.array([counts[i] for i in range(dm.num_classes)])
nice=[c.replace('_',' ') for c in classes]

## 2 · Headline metrics

In [ ]:
acc=accuracy_score(y_true,y_pred); mf1=f1_score(y_true,y_pred,average='macro')
wf1=f1_score(y_true,y_pred,average='weighted')
def topk(pr,y,k): tk=np.argsort(pr,1)[:,-k:]; return np.mean([y[i] in tk[i] for i in range(len(y))])
t3=topk(probs,y_true,3)

fig=make_subplots(rows=1,cols=4,specs=[[{'type':'indicator'}]*4],horizontal_spacing=0.06)
def ind(v,title,col,ref=None):
    return go.Indicator(mode="gauge+number",value=v*100,number={'suffix':'%','font':{'size':30}},
        gauge={'axis':{'range':[0,100],'tickcolor':SUB},'bar':{'color':col,'thickness':0.72},
               'bgcolor':PANEL,'borderwidth':0},title={'text':title,'font':{'size':14,'color':SUB}})
for i,(v,t,c) in enumerate([(acc,'Accuracy',TEAL),(mf1,'Macro-F1',CORAL),
                            (wf1,'Weighted-F1',GOLD),(t3,'Top-3 accuracy',PLUM)]):
    fig.add_trace(ind(v,t,c),row=1,col=i+1)
fig.update_layout(height=260,title="Champion model — headline metrics",
                  margin=dict(l=20,r=20,t=80,b=20))
fig.show()
print(f'accuracy {acc:.3f} · macro-F1 {mf1:.3f} · weighted-F1 {wf1:.3f} · top-3 {t3:.3f}')
print(f'macro↔weighted gap {abs(wf1-mf1)*100:.1f} pts (larger = thin classes drag macro down)')

## 3 · Per-class F1 — sorted (hover for precision/recall/support)

In [ ]:
order=np.argsort(f1c)
supp=np.array([int((y_true==i).sum()) for i in range(dm.num_classes)])
hov=[f"<b>{nice[i]}</b><br>F1 {f1c[i]:.2f}<br>precision {prec[i]:.2f}"
     f"<br>recall {rec[i]:.2f}<br>test imgs {supp[i]}" for i in order]
fig=go.Figure(go.Bar(
    x=[f1c[i] for i in order], y=[nice[i] for i in order], orientation='h',
    marker=dict(color=[qcolor(f1c[i]) for i in order],
                line=dict(color='rgba(0,0,0,0.06)',width=1)),
    text=[f"{f1c[i]:.2f}" for i in order], textposition='outside',
    hovertext=hov, hoverinfo='text'))
fig.add_vline(x=mf1,line=dict(color=SUB,dash='dash'),
              annotation_text=f"macro-F1 {mf1:.2f}",annotation_position="top")
fig.update_layout(height=720,title="Per-class F1 (sorted worst→best)",
                  xaxis_title="F1 score",xaxis_range=[0,1.08],
                  yaxis=dict(tickfont=dict(size=11)))
fig.show()

## 4 · F1 vs training-set size — the *data-limited* evidence
Each dot is a site. Hover for names. If weak dots sit at low photo counts, performance is limited by **data quantity**.

In [ ]:
z=np.polyfit(sizes,f1c,1); xs=np.linspace(sizes.min(),sizes.max(),60)
fig=go.Figure()
fig.add_trace(go.Scatter(x=xs,y=np.poly1d(z)(xs),mode='lines',
    line=dict(color=SUB,dash='dash'),name='trend',hoverinfo='skip'))
fig.add_trace(go.Scatter(x=sizes,y=f1c,mode='markers',name='sites',
    marker=dict(size=15,color=f1c,colorscale=[[0,QUAL[0]],[0.6,QUAL[1]],[0.8,QUAL[2]],[1,QUAL[3]]],
                cmin=0,cmax=1,line=dict(color='white',width=1.5),
                colorbar=dict(title='F1',thickness=14)),
    text=nice,hovertemplate="<b>%{text}</b><br>%{x} imgs · F1 %{y:.2f}<extra></extra>"))
# annotate only the weak tail (avoids overlap; rest is hover)
for i in range(dm.num_classes):
    if f1c[i]<0.7:
        fig.add_annotation(x=sizes[i],y=f1c[i],text=nice[i],showarrow=True,arrowcolor=SUB,
            arrowwidth=1,ax=24,ay=-22,font=dict(size=10,color=INK),
            bgcolor="rgba(255,255,255,0.85)",bordercolor=GRID,borderwidth=1)
fig.add_vline(x=20,line=dict(color=CORAL,dash='dot'),
              annotation_text="~20-photo line",annotation_position="bottom")
fig.update_layout(height=520,title="Performance vs data quantity",
    xaxis_title="training images for the class",yaxis_title="test F1",yaxis_range=[-0.05,1.1])
fig.show()
print(f'correlation(F1, size) = {np.corrcoef(sizes,f1c)[0,1]:.2f}  (positive ⇒ more data, higher F1)')

## 5 · Precision vs Recall (hover for names)
Bottom-right = a *magnet* (over-predicted). Top-left = *escapee* (its photos leak away). Top-right = healthy.

In [ ]:
fig=go.Figure()
fig.add_shape(type='line',x0=0,y0=0,x1=1,y1=1,line=dict(color=GRID,dash='dash'))
fig.add_trace(go.Scatter(x=rec,y=prec,mode='markers',
    marker=dict(size=15,color=f1c,colorscale=[[0,QUAL[0]],[0.6,QUAL[1]],[0.8,QUAL[2]],[1,QUAL[3]]],
                cmin=0,cmax=1,line=dict(color='white',width=1.5),
                colorbar=dict(title='F1',thickness=14)),
    text=nice,hovertemplate="<b>%{text}</b><br>precision %{y:.2f}<br>recall %{x:.2f}<extra></extra>"))
for i in range(dm.num_classes):
    if prec[i]<0.7 or rec[i]<0.7:
        fig.add_annotation(x=rec[i],y=prec[i],text=nice[i],showarrow=True,arrowcolor=SUB,
            arrowwidth=1,ax=0,ay=-24,font=dict(size=9,color=INK),
            bgcolor="rgba(255,255,255,0.85)",bordercolor=GRID,borderwidth=1)
fig.update_layout(height=560,title="Precision vs Recall per site",
    xaxis_title="recall",yaxis_title="precision",xaxis_range=[-0.05,1.08],yaxis_range=[-0.05,1.08])
fig.show()

## 6 · Confusion matrix — interactive (hover shows exact true → predicted)

In [ ]:
cm=confusion_matrix(y_true,y_pred,labels=range(dm.num_classes))
cmn=np.nan_to_num(cm/cm.sum(1,keepdims=True))
fig=go.Figure(go.Heatmap(z=cmn,x=nice,y=nice,colorscale='Teal',zmin=0,zmax=1,
    hovertemplate="true <b>%{y}</b><br>predicted <b>%{x}</b><br>%{z:.0%} of true<extra></extra>",
    colorbar=dict(title='row %',thickness=14)))
fig.update_layout(height=760,title="Confusion matrix (row-normalized)",
    xaxis=dict(title='Predicted',tickangle=-55,tickfont=dict(size=9)),
    yaxis=dict(title='True',tickfont=dict(size=9),autorange='reversed'),
    margin=dict(l=140,b=140))
fig.show()

## 7 · Confidence — does the model know when it's unsure?
Distribution of the top-prediction confidence, split by whether the prediction was correct.

In [ ]:
correct=y_true==y_pred
fig=go.Figure()
fig.add_trace(go.Histogram(x=pred_conf[correct],name='correct',opacity=0.75,
    marker_color=TEAL,xbins=dict(start=0,end=1,size=0.05)))
fig.add_trace(go.Histogram(x=pred_conf[~correct],name='wrong',opacity=0.75,
    marker_color=CORAL,xbins=dict(start=0,end=1,size=0.05)))
fig.update_layout(height=430,barmode='overlay',title="Confidence: correct vs wrong predictions",
    xaxis_title="top prediction confidence",yaxis_title="count")
fig.show()
print(f'mean confidence — correct {pred_conf[correct].mean():.1%} · wrong {pred_conf[~correct].mean():.1%}')

## 8 · Selective prediction — the "I don't know" trade-off
Answer only when confidence ≥ threshold. See how coverage trades against accuracy.

In [ ]:
ths=np.linspace(0.1,0.95,25); cov=[]; sa=[]
for t in ths:
    k=pred_conf>=t; cov.append(k.mean())
    sa.append(accuracy_score(y_true[k],y_pred[k]) if k.any() else np.nan)
fig=go.Figure()
fig.add_trace(go.Scatter(x=ths,y=cov,mode='lines+markers',name='coverage (% answered)',
    line=dict(color=SKY,width=3),hovertemplate="thr %{x:.2f}<br>coverage %{y:.0%}<extra></extra>"))
fig.add_trace(go.Scatter(x=ths,y=sa,mode='lines+markers',name='accuracy on answered',
    line=dict(color=CORAL,width=3),hovertemplate="thr %{x:.2f}<br>sel-acc %{y:.0%}<extra></extra>"))
fig.add_hline(y=acc,line=dict(color=SUB,dash='dash'),
    annotation_text=f"base accuracy {acc:.0%}",annotation_position="bottom right")
fig.update_layout(height=430,title="Selective prediction trade-off",
    xaxis_title="confidence threshold",yaxis_title="fraction",yaxis_range=[0,1.05])
fig.show()

## 9 · Fine-tuning strategy comparison

In [ ]:
strat=['Frozen (phase 1)','Partial (last 2 blocks)','Full (phase 2 champion)']
acc_s=[0.799,0.827,0.866]; f1_s=[0.784,0.806,0.858]; tp=[1.7,62.7,100]
fig=go.Figure()
fig.add_trace(go.Bar(x=strat,y=acc_s,name='test accuracy',marker_color=TEAL,
    text=[f"{v:.1%}" for v in acc_s],textposition='outside'))
fig.add_trace(go.Bar(x=strat,y=f1_s,name='macro-F1',marker_color=CORAL,
    text=[f"{v:.1%}" for v in f1_s],textposition='outside'))
for i,t in enumerate(tp):
    fig.add_annotation(x=strat[i],y=0.04,text=f"{t:.0f}% trainable",showarrow=False,
                       font=dict(size=11,color=SUB))
fig.update_layout(height=470,barmode='group',title="Accuracy rises with adaptation",
    yaxis_title="score",yaxis_range=[0,1.0])
fig.show()

---
### Reading this dashboard
- **§2** headline: ~86.6% accuracy / ~85.8% macro-F1.
- **§4** is the key evidence: weak sites cluster at low photo counts ⇒ **data-limited**.
- **§5** separates *magnets* (low precision) from *escapees* (low recall).
- **§7–8** show the model is calibrated enough for a usable confidence threshold ("I don't know").
- **§9** proves full fine-tuning beat frozen and partial — with evidence, not assumption.

*Tip: hover any point for its site name — that's why labels aren't crowded onto the plots.*